# RAG vs Long Context — Benchmark

**Goal:** Run the same queries against a RAG pipeline and a long-context approach. Compare what the LLM receives, measure token usage and cost, and see when each approach wins.

**API key optional** — Core comparison works without an API key. With an OpenAI key, you'll also see generated answers and latency measurements.

---

## What You'll Learn

1. How RAG and long context assemble prompts differently
2. The token usage and cost gap at scale
3. When each approach produces better results
4. Why the hybrid approach is the production standard

In [ ]:
# Uncomment to install
# !pip install chromadb openai sentence-transformers python-dotenv tiktoken

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import chromadb
import tiktoken
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = tiktoken.get_encoding('cl100k_base')

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    from openai import OpenAI
    llm_client = OpenAI(api_key=api_key)
    HAS_LLM = True
    print("✓ Embedding model: all-MiniLM-L6-v2 (local)")
    print("✓ LLM: OpenAI GPT-4o-mini (API)")
else:
    HAS_LLM = False
    print("✓ Embedding model: all-MiniLM-L6-v2 (local)")
    print("⚠️ LLM: Not available (no API key). Prompt comparison still works.")

print("\nSetup complete!")

## Step 1: Prepare the Corpus

Same consulting firm Q3 data — used as a single block for long context and chunked for RAG.

In [ ]:
# The full corpus (all Q3 reports combined)
documents = [
    {"title": "Q3 Financial Summary", "content": """Total revenue for Q3 2025 reached $42.3 million, representing a 12% increase year-over-year and a 6% decrease from Q2 2025 ($45.1 million). The decline from Q2 is attributed primarily to the conclusion of two large federal contracts and seasonal slowdown in the financial services practice. Revenue by practice area: Defense & Security $15.2M (36% of total, +18% YoY), Financial Services $12.8M (30% of total, +5% YoY), Health $8.1M (19% of total, +15% YoY), Energy & Infrastructure $6.2M (15% of total, +3% YoY). Gross margin improved to 38.5%, up from 36.2% in Q2, driven by higher utilization of senior consultants and favorable project mix. Operating expenses were held flat at $12.1 million despite headcount growth. The firm's backlog stands at $127 million, providing approximately 9 months of revenue visibility. New bookings in Q3 totaled $38 million, with a book-to-bill ratio of 0.90."""},
    {"title": "Q3 People Report", "content": """Total headcount reached 852 employees at the end of Q3, a net increase of 28 from Q2. The firm hired 45 new consultants during the quarter, with 17 departures (voluntary attrition rate of 8% annualized, down from 11% in Q3 2024). Average time-to-fill was 34 days with an offer acceptance rate of 87%. Consultant utilization averaged 78% in Q3, down from 82% in Q2. The decline reflects the onboarding ramp for new hires and seasonal slowdown. Employee engagement score: 4.1/5.0 (unchanged from Q2). The offshore team in India grew to 45 members."""},
    {"title": "Q3 Client Report", "content": """The firm onboarded 12 new clients in Q3, bringing the active client count to 94. Client retention rate remains strong at 92%. Notable wins: Department of Defense $8.2M multi-year AI modernization contract, major regional bank $3.1M regulatory compliance, state health agency $2.4M Medicaid modernization, Fortune 500 insurer $1.8M analytics. Client satisfaction scores averaged 4.7/5.0, the highest quarterly score in firm history. Net Promoter Score improved to 62, up from 58 in Q2. Average deal size decreased to $350K from $563K in Q2. Expansion revenue from 8 existing clients contributed $14.2 million (34% of total Q3 revenue). Pipeline: 47 qualified opportunities worth $89 million."""},
    {"title": "Q3 Technology Report", "content": """The internal AI analytics platform is now used by 73% of senior leadership (up from 45% in Q2). GenAI dashboard processes 340 queries per day with 94% satisfaction. Common queries: revenue breakdown (28%), utilization (22%), pipeline (19%), headcount (15%). Azure migration 85% complete. Cloud costs 15% below budget. AI-related revenue: $6.8M (16% of Q3 total), up from $3.2M in Q3 2024 (113% YoY growth). FedRAMP authorization maintained, zero security incidents, clean SOC 2 Type II audit."""}
]

# Combine for long context
full_corpus = "\n\n".join([f"[{d['title']}]\n{d['content']}" for d in documents])
corpus_tokens = len(tokenizer.encode(full_corpus))

print(f"📄 Corpus: {len(documents)} documents, {corpus_tokens:,} tokens total")
for doc in documents:
    t = len(tokenizer.encode(doc['content']))
    print(f"   • {doc['title']}: {t} tokens")

In [ ]:
# Set up RAG pipeline
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="benchmark")

# Chunk by paragraph and embed
all_chunks = []
for doc in documents:
    paragraphs = [p.strip() for p in doc['content'].split('. ') if len(p.strip()) > 50]
    # Re-join into meaningful chunks (2-3 sentences each)
    for i in range(0, len(paragraphs), 2):
        chunk_text = '. '.join(paragraphs[i:i+2]) + '.'
        all_chunks.append({'text': chunk_text, 'title': doc['title']})

texts = [c['text'] for c in all_chunks]
embeddings = embedding_model.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    metadatas=[{'title': c['title']} for c in all_chunks],
)

print(f"✓ RAG pipeline ready: {collection.count()} chunks in ChromaDB")

## Step 2: Define Test Queries

Three categories that test different strengths:
- **Factual lookup** — Find a specific number (RAG should excel)
- **Cross-document** — Connect information across reports (hybrid shines)
- **Full synthesis** — Summarize everything (long context should excel)

In [ ]:
test_queries = [
    {"type": "Factual Lookup",     "query": "What is the book-to-bill ratio for Q3?"},
    {"type": "Factual Lookup",     "query": "What was the client satisfaction score?"},
    {"type": "Cross-Document",     "query": "How does utilization relate to the revenue decline from Q2?"},
    {"type": "Cross-Document",     "query": "What is the connection between AI revenue growth and technology adoption?"},
    {"type": "Full Synthesis",     "query": "Summarize the firm's overall Q3 performance across all areas."},
    {"type": "Full Synthesis",     "query": "What are the biggest risks and opportunities heading into Q4?"},
]

print(f"📋 {len(test_queries)} test queries across 3 categories:")
for q in test_queries:
    print(f"   [{q['type']}] {q['query']}")

## Step 3: Run Both Approaches

For each query, we build the prompt both ways and compare token usage. With an API key, we also compare generated answers and latency.

In [ ]:
system_prompt = """You are a senior analyst for a consulting firm's C-suite dashboard.
Answer based ONLY on the provided context. Cite specific numbers.
If the data isn't available, say so. Be concise."""

results = []

for q in test_queries:
    query = q['query']
    
    # --- RAG approach ---
    query_emb = embedding_model.encode([query]).tolist()
    rag_results = collection.query(query_embeddings=query_emb, n_results=3)
    rag_context = "\n\n".join([
        f"[{rag_results['metadatas'][0][i]['title']}] {rag_results['documents'][0][i]}"
        for i in range(3)
    ])
    rag_prompt = f"CONTEXT:\n{rag_context}\n\nQUESTION: {query}"
    rag_tokens = len(tokenizer.encode(system_prompt + rag_prompt))
    
    # --- Long context approach ---
    lc_prompt = f"CONTEXT:\n{full_corpus}\n\nQUESTION: {query}"
    lc_tokens = len(tokenizer.encode(system_prompt + lc_prompt))
    
    result = {
        'type': q['type'],
        'query': query,
        'rag_tokens': rag_tokens,
        'lc_tokens': lc_tokens,
        'rag_context': rag_context,
    }
    
    # Generate answers if LLM available
    if HAS_LLM:
        start = time.time()
        rag_response = llm_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": rag_prompt}],
            temperature=0.1, max_tokens=300,
        )
        result['rag_latency'] = time.time() - start
        result['rag_answer'] = rag_response.choices[0].message.content
        
        start = time.time()
        lc_response = llm_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": lc_prompt}],
            temperature=0.1, max_tokens=300,
        )
        result['lc_latency'] = time.time() - start
        result['lc_answer'] = lc_response.choices[0].message.content
    
    results.append(result)

print(f"✓ Ran {len(results)} queries through both approaches")

## Step 4: Compare Results

In [ ]:
for r in results:
    print(f"\n{'='*70}")
    print(f"🔎 [{r['type']}] {r['query']}")
    print(f"{'-'*70}")
    print(f"  Token usage:  RAG = {r['rag_tokens']:,}  |  Long Context = {r['lc_tokens']:,}  |  Ratio: {r['lc_tokens']/r['rag_tokens']:.1f}x")
    
    if HAS_LLM:
        print(f"  Latency:      RAG = {r['rag_latency']:.2f}s  |  Long Context = {r['lc_latency']:.2f}s")
        print(f"\n  📗 RAG Answer:")
        print(f"  {r['rag_answer'][:200]}")
        print(f"\n  📘 Long Context Answer:")
        print(f"  {r['lc_answer'][:200]}")
    else:
        print(f"\n  📗 RAG would send {r['rag_tokens']} tokens (3 relevant chunks)")
        print(f"  📘 Long Context would send {r['lc_tokens']} tokens (entire corpus)")

## Step 5: Visualize the Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [f"Q{i+1}" for i in range(len(results))]
rag_tokens = [r['rag_tokens'] for r in results]
lc_tokens = [r['lc_tokens'] for r in results]

# Token usage comparison
x = np.arange(len(labels))
width = 0.35
axes[0].bar(x - width/2, rag_tokens, width, label='RAG', color='#2ECC71')
axes[0].bar(x + width/2, lc_tokens, width, label='Long Context', color='#3498DB')
axes[0].set_ylabel('Tokens per Query')
axes[0].set_title('Token Usage: RAG vs Long Context', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Cost projection at scale
queries_per_day = [10, 100, 500, 1000, 5000, 10000]
# GPT-4o-mini pricing: $0.15 per 1M input tokens
price_per_token = 0.15 / 1_000_000
avg_rag = np.mean(rag_tokens)
avg_lc = np.mean(lc_tokens)

rag_daily_cost = [q * avg_rag * price_per_token for q in queries_per_day]
lc_daily_cost = [q * avg_lc * price_per_token for q in queries_per_day]

axes[1].plot(queries_per_day, rag_daily_cost, 'o-', color='#2ECC71', label='RAG', linewidth=2)
axes[1].plot(queries_per_day, lc_daily_cost, 'o-', color='#3498DB', label='Long Context', linewidth=2)
axes[1].set_xlabel('Queries per Day')
axes[1].set_ylabel('Daily Cost ($)')
axes[1].set_title('Projected Daily Cost at Scale', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

ratio = avg_lc / avg_rag
print(f"\n📊 Average tokens per query: RAG = {avg_rag:.0f}, Long Context = {avg_lc:.0f}")
print(f"   Long context uses {ratio:.1f}x more tokens per query.")
print(f"\n   At 1,000 queries/day: RAG ≈ ${1000 * avg_rag * price_per_token:.2f}/day, Long Context ≈ ${1000 * avg_lc * price_per_token:.2f}/day")

## Step 6: When Each Approach Wins

Let's look at the results by query type to understand the trade-offs.

In [ ]:
print("When Each Approach Wins")
print("=" * 60)

print("\n📗 RAG excels at:")
print("   • Factual lookups — retrieves the exact chunk with the answer")
print("   • Specific numbers — 'What is the book-to-bill ratio?'")
print("   • Speed — processes only relevant tokens")
print("   • Cost — at scale, dramatically cheaper per query")

print("\n📘 Long Context excels at:")
print("   • Cross-document synthesis — sees all data simultaneously")
print("   • Full summaries — 'Summarize overall Q3 performance'")
print("   • Complex relationships — connecting dots across reports")
print("   • Simplicity — no chunking, embedding, or retrieval pipeline")

print("\n🏆 Hybrid wins for production:")
print("   • Use RAG to identify relevant sections")
print("   • Load those sections (with surrounding context) into a generous window")
print("   • Get RAG's precision with long context's reasoning depth")

# Summary table
print("\n" + "-" * 60)
print(f"{'Query Type':<20} {'RAG Strength':<20} {'Long Context Strength'}")
print(f"{'-'*20} {'-'*20} {'-'*20}")
print(f"{'Factual Lookup':<20} {'⭐ Precise':<20} {'Good but overkill'}")
print(f"{'Cross-Document':<20} {'May miss links':<20} {'⭐ Sees everything'}")
print(f"{'Full Synthesis':<20} {'Fragmented view':<20} {'⭐ Complete picture'}")

## Key Takeaways

1. **RAG uses dramatically fewer tokens.** For the same query, RAG sends 5-20x fewer tokens to the LLM. This translates directly to lower cost and faster responses.

2. **Long context is better for synthesis.** When you need the LLM to reason across all your data simultaneously — summaries, trend analysis, cross-document connections — long context gives it the complete picture.

3. **RAG can miss connections.** If the retrieval step doesn't find a relevant chunk, the LLM never sees that data. Long context doesn't have this blind spot (but has the "lost in the middle" problem instead).

4. **Cost diverges exponentially at scale.** The per-query difference seems small, but multiply by thousands of queries per day and long context becomes orders of magnitude more expensive.

5. **The hybrid approach is the answer.** Use RAG for retrieval precision, long context for reasoning depth. This is what the consulting firm's C-suite dashboard uses in production.

---

### Next Steps
- **[RAG vs Long Context (doc)](../docs/05-rag-vs-long-context.md)** — Full analysis of trade-offs and decision frameworks
- **[02-build-a-rag-pipeline.ipynb](02-build-a-rag-pipeline.ipynb)** — Build the full RAG system
- **[03-chunking-strategies.ipynb](03-chunking-strategies.ipynb)** — Optimize the chunking that feeds the RAG pipeline